In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix, classification_report
import joblib

In [2]:
df= pd.read_excel("../data/cleaned_data.xlsx")

In [3]:
df["Investment_Return"]= (df["Price_After_5_Years"]-df["Current_Price"])/df["Current_Price"]

In [4]:
df["Investment_Return"].describe()

count    500.000000
mean       0.368024
std        0.125914
min        0.159269
25%        0.261075
50%        0.362098
75%        0.469492
max        0.610100
Name: Investment_Return, dtype: float64

In [5]:
df[df["Investment_Return"] >= 0.30].describe()


,Property_ID,Area_sqft,Bedrooms,Bathrooms,Floors,Location_Score,School_Score,Current_Price,Price_Year_1,Price_Year_2,Price_Year_3,Price_Year_4,Price_After_5_Years,Price_per_sqft,Area_per_Bedroom,Property_age,Investment_Return
count,326.000000,326.000000,326.000000,326.000000,326.000000,326.000000,326.000000,3.260000e+02,3.260000e+02,3.260000e+02,3.260000e+02,3.260000e+02,3.260000e+02,326.000000,326.000000,326.000000,326.000000
mean,249.463190,2973.932515,2.966258,2.453988,1.840491,76.573620,69.371166,6.601489e+05,7.100114e+05,7.637544e+05,8.216882e+05,8.841489e+05,9.515004e+05,245.102544,1355.070501,24.638037,0.441374
std,144.711714,1198.796228,1.384117,1.102179,0.825936,14.551525,17.581992,1.565705e+05,1.686232e+05,1.821232e+05,1.972562e+05,2.142263e+05,2.332571e+05,66.854158,1081.241381,12.818891,0.089105
min,4.000000,802.000000,1.000000,1.000000,1.000000,50.000000,40.000000,2.956020e+05,3.134080e+05,3.322860e+05,3.523010e+05,3.735220e+05,3.960210e+05,165.050836,196.000000,3.000000,0.300564
25%,126.250000,1904.750000,2.000000,1.000000,1.000000,63.000000,54.250000,5.322055e+05,5.733715e+05,6.173852e+05,6.669412e+05,7.146998e+05,7.612295e+05,195.669803,663.150000,14.000000,0.367334
50%,247.500000,2998.000000,3.000000,2.000000,2.000000,78.000000,68.000000,6.620295e+05,7.147185e+05,7.700705e+05,8.252865e+05,8.763110e+05,9.418615e+05,223.959765,983.700000,23.500000,0.435061
75%,374.500000,3993.750000,4.000000,3.000000,3.000000,89.000000,85.000000,7.869998e+05,8.453500e+05,9.111395e+05,9.826135e+05,1.059876e+06,1.140610e+06,277.330324,1600.750000,36.000000,0.515848
max,499.000000,4973.000000,5.000000,4.000000,3.000000,100.000000,100.000000,1.010002e+06,1.089094e+06,1.174380e+06,1.266345e+06,1.379349e+06,1.511775e+06,523.576060,4937.000000,46.000000,0.610100


In [6]:
df["Good_Investment"]= (df["Investment_Return"] >= 0.30).astype(int)

In [7]:
df["Good_Investment"].value_counts()

Good_Investment
1    326
0    174
Name: count, dtype: int64

In [8]:
df

,Property_ID,Area_sqft,Bedrooms,Bathrooms,Floors,Location_Score,School_Score,Current_Price,Price_Year_1,Price_Year_2,Price_Year_3,Price_Year_4,Price_After_5_Years,Price_Category,Price_per_sqft,Area_per_Bedroom,Property_age,Investment_Return,Good_Investment
0,1,2751,3,4,3,54,58,624636,656833,690689,726290,763726,803092,Medium,227.057797,917.000000,37,0.285696,0
1,2,3322,5,1,1,62,78,694237,729166,765853,804385,844856,887363,Medium,208.981638,664.400000,7,0.278185,0
2,3,4765,3,3,2,57,50,771778,803688,836917,871520,907554,945078,Medium,161.968101,1588.333333,37,0.224546,0
3,4,4512,2,2,3,57,90,792597,854577,921404,993457,1071144,1154906,High,175.664229,2256.000000,22,0.457116,1
4,5,2436,3,1,1,76,69,580257,612034,645552,680905,718194,757525,Low,238.200739,812.000000,39,0.305499,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,1563,5,1,2,72,69,487618,516856,547847,580696,615515,652422,Low,311.975688,312.600000,8,0.337978,1
496,497,4682,4,1,2,69,96,934003,997429,1065162,1137495,1214740,1297231,High,199.488039,1170.500000,44,0.388894,1
497,498,4205,3,4,1,96,52,821328,855209,890488,927222,965472,1005300,Medium,195.321760,1401.666667,37,0.223993,0
498,499,3413,5,2,1,96,63,746551,806353,870945,940712,1016067,1097459,High,218.737474,682.600000,28,0.470039,1


In [9]:
X= df.drop(columns=["Good_Investment", "Investment_Return", "Price_After_5_Years", "Price_Year_1", "Price_Year_2", "Price_Year_3", "Price_Year_4", "Price_Category","Property_ID"])

In [10]:
y= df["Good_Investment"]

In [11]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

In [12]:
scaler_cls= StandardScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

In [13]:
models={
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Decision tree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier()
}
results= []
for model_name, model in models.items():
    if model_name == "Logistic Regression":
        model.fit(X_train_cls_scaled, y_train_cls)
        y_pred= model.predict(X_test_cls_scaled)
    else:
        model.fit(X_train_cls, y_train_cls)
        y_pred= model.predict(X_test_cls)
    accuracy= accuracy_score(y_test_cls, y_pred)
    Precision= precision_score(y_test_cls, y_pred)
    Recall= recall_score(y_test_cls, y_pred)
    F1_Score= f1_score(y_test_cls, y_pred)
    f1_macro= f1_score(y_test_cls, y_pred, average="macro")
    balanced_accuracy= balanced_accuracy_score(y_test_cls, y_pred)
    results.append({"Model": model_name, "Accuracy": accuracy, "Precision": Precision, "Recall": Recall, "F1_Score": F1_Score, "F1_Macro": f1_macro, "Balanced_Accuracy": balanced_accuracy})

In [14]:
pd.DataFrame(results)

,Model,Accuracy,Precision,Recall,F1_Score,F1_Macro,Balanced_Accuracy
0,Logistic Regression,0.64,0.646465,0.984615,0.780488,0.390244,0.492308
1,Random Forest,0.57,0.627907,0.830769,0.715232,0.418840,0.458242
2,Gradient Boosting,0.60,0.654321,0.815385,0.726027,0.492643,0.507692
3,Decision tree,0.61,0.675676,0.769231,0.719424,0.540040,0.541758
4,XGBoost,0.63,0.679487,0.815385,0.741259,0.546068,0.550549


In [15]:
param_grid_cls= {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1]
}
grid_search_cls= GridSearchCV(
    XGBClassifier(), 
    param_grid_cls, 
    cv=5, 
    scoring="f1_macro"
)

In [16]:
grid_search_cls.fit(X_train_cls, y_train_cls)
grid_search_cls.best_params_

{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}

In [17]:
grid_search_cls.best_estimator_

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
y_pred_tuned = grid_search_cls.best_estimator_.predict(X_test_cls)

In [19]:
accuracy= accuracy_score(y_test_cls, y_pred_tuned)
precision= precision_score(y_test_cls, y_pred_tuned)
recall= recall_score(y_test_cls, y_pred_tuned)
f1= f1_score(y_test_cls, y_pred_tuned)
f1_macro= f1_score(y_test_cls, y_pred_tuned, average="macro")
balanced_accuracy= balanced_accuracy_score(y_test_cls, y_pred_tuned)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")

Accuracy: 0.6000
Precision: 0.6543
Recall: 0.8154
F1-Score: 0.7260
F1 Macro: 0.4926
Balanced Accuracy: 0.5077


In [20]:
confusion_matrix(y_test_cls, y_pred_tuned)

array([[ 7, 28],
       [12, 53]])

In [21]:
final_model= XGBClassifier()
final_model.fit(X_train_cls, y_train_cls)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [22]:
y_pred_final= final_model.predict(X_test_cls)
accuracy= accuracy_score(y_test_cls, y_pred_final)
precision= precision_score(y_test_cls, y_pred_final)
recall= recall_score(y_test_cls, y_pred_final)
f1= f1_score(y_test_cls, y_pred_final)
f1_macro= f1_score(y_test_cls, y_pred_final, average="macro")
balanced_accuracy= balanced_accuracy_score(y_test_cls, y_pred_final)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")

Accuracy: 0.6300
Precision: 0.6795
Recall: 0.8154
F1-Score: 0.7413
F1 Macro: 0.5461
Balanced Accuracy: 0.5505


In [23]:
confusion_matrix(y_test_cls, y_pred_final)

array([[10, 25],
       [12, 53]])

In [24]:
joblib.dump(final_model, "../models/xgb_classifier.pkl")
joblib.dump(scaler_cls, "../models/scaler_cls.pkl")

['../models/scaler_cls.pkl']